# DRCFNet Model - CMU-MOSI Dataset

Dynamic Reconstructive Context Fusion Network for Sentiment Analysis

In [ ]:
!git clone https://github.com/ranbeer06052009/MulG-Research

Cloning into 'MulG-Research'...
remote: Enumerating objects: 232, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (85/85), done.


In [ ]:
import gdown

file_id = "1szKIqO0t3Be_W91xvf6aYmsVVUa7wDHU"
destination = "mosi_raw.pkl"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}", destination, quiet=False)

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.append('/content/MulG-Research/src')

from loader import get_dataloader
from models.drcfnet import DRCFNet
from training.drcfnet_loss import DRCFNetLoss
from training.drcfnet_train import train, test
from evaluation.performance import eval_affect

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

In [ ]:
# Load MOSI Data
train_data, valid_data, test_data = get_dataloader(
    '/content/mosi_raw.pkl',
    max_pad=True,
    max_seq_len=50
)

In [ ]:
# Initialize Model and Loss
model = DRCFNet(dim_v=35, dim_a=74, dim_t=300, d=128, n_heads=4, dropout=0.2).to(device)
criterion = DRCFNetLoss(lambda_orth=0.01, lambda_contrastive=0.01, task_weight=1.0).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

In [ ]:
# Train the model
model = train(
    model=model,
    train_loader=train_data,
    valid_loader=valid_data,
    criterion=criterion,
    optimizer=optimizer,
    epochs=100,
    device=device
)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
# Test the model
test_loss, preds, labels = test(
    model=model,
    dataloader=test_data,
    criterion=criterion,
    device=device,
    return_preds=True
)

preds_np = preds.view(-1).cpu().numpy()
labels_np = labels.view(-1).cpu().numpy()

# ===== MAE =====
mae = np.mean(np.abs(preds_np - labels_np))

# ===== Corr =====
corr = np.corrcoef(preds_np, labels_np)[0][1]
corr = 0 if np.isnan(corr) else corr

# ===== Binary =====
pred_bin = (preds_np > 0).astype(int)
label_bin = (labels_np > 0).astype(int)

mask = labels_np != 0
acc2 = (pred_bin[mask] == label_bin[mask]).mean()

f1 = f1_score(label_bin[mask], pred_bin[mask], average='weighted')

print("\nFinal Evaluation:")
print(f"MAE: {mae:.4f}")
print(f"Corr: {corr:.4f}")
print(f"Acc-2: {acc2*100:.2f}%")
print(f"F1: {f1*100:.2f}%")